In [1]:
# 1. Install Python 3.10 and venv support
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y

# 2. Create an isolated virtual environment
!python3.10 -m venv /content/venv

# 3. Upgrade pip
!/content/venv/bin/python -m pip install --upgrade pip

# 4. Install the official W3D4 pins
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "autoawq==0.2.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"

print("W3D4 virtual environment ready!")

Get:1 https://cli.github.com/packages stable InRelease [4,685 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,920 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ub

In [2]:
import subprocess, sys

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

print("W3D4 pins loaded")

W3D4 pins loaded


In [3]:
!/content/venv/bin/python -m pip install -q \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*" \
    "autoawq==0.2.*"

print("W3D4 serving pins + AutoAWQ installed")

W3D4 serving pins + AutoAWQ installed


In [4]:
import os
import signal
import subprocess
import sys

MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

cmd = [
    "/content/venv/bin/python",
    "-m", "vllm.entrypoints.openai.api_server"
]

for k, v in SERVER_ARGS.items():
    if v is None:
        cmd.append(k)
    else:
        cmd += [k, str(v)]

print("Launching AWQ server...")
print(" ".join(cmd))

logf = open(SERVER_LOG, "wb")

server = subprocess.Popen(
    cmd,
    stdout=logf,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print(f"AWQ server started. PID: {server.pid}")
print(f"Logs: {SERVER_LOG}")

Launching AWQ server...
/content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
AWQ server started. PID: 5262
Logs: /content/server.log


In [5]:
import time
import urllib.request
import urllib.error

url = "http://localhost:8000/v1/models"
timeout_s = 300
deadline = time.time() + timeout_s

while time.time() < deadline:
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            if r.status == 200:
                print("SERVER HEALTHY:", url, "->", r.status)
                break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass

    time.sleep(3)
else:
    print("TIMED OUT")
    print("Check server.log:")
    with open("/content/server.log", "r", errors="replace") as f:
        print("".join(f.readlines()[-30:]))

SERVER HEALTHY: http://localhost:8000/v1/models -> 200


In [6]:
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

11723 MiB


In [7]:
from openai import OpenAI
import time

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed",
)

prompt = "Explain in simple terms what an inference server does."

start = time.time()

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200,
)

elapsed = time.time() - start
text = response.choices[0].message.content
tokens = response.usage.completion_tokens

print("Completion tokens:", tokens)
print("Elapsed time:", round(elapsed, 2), "seconds")
print("Tokens/s:", round(tokens / elapsed, 2))
print("\nResponse:")
print(text)

Completion tokens: 93
Elapsed time: 2.76 seconds
Tokens/s: 33.71

Response:
An inference server is like a smart assistant that listens to your questions and provides answers or solutions about certain topics. It uses the knowledge it learned to figure out a response across a given number of steps, often waiting for more input from users for guidance. While the server gets more feedback about a user's context, it makes more informed decisions for simple queries. The larger goal of the server is to provide useful responses while being efficient in how it accomplishes this.


In [8]:
import os
import signal
import time

os.killpg(os.getpgid(server.pid), signal.SIGTERM)

time.sleep(3)

print("AWQ server stopped.")

AWQ server stopped.


In [9]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
}

cmd = [
    "/content/venv/bin/python",
    "-m", "vllm.entrypoints.openai.api_server"
]

for k, v in SERVER_ARGS.items():
    if v is None:
        cmd.append(k)
    else:
        cmd += [k, str(v)]

print("Launching FP16 server...")
print(" ".join(cmd))

logf = open("/content/server.log", "wb")

server = subprocess.Popen(
    cmd,
    stdout=logf,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print(f"FP16 server started. PID: {server.pid}")

Launching FP16 server...
/content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000
FP16 server started. PID: 6117


In [10]:
import time
import urllib.request
import urllib.error

url = "http://localhost:8000/v1/models"
timeout_s = 300
deadline = time.time() + timeout_s

while time.time() < deadline:
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            if r.status == 200:
                print("FP16 SERVER HEALTHY:", url, "->", r.status)
                break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass

    time.sleep(3)
else:
    print("TIMED OUT")
    with open("/content/server.log", "r", errors="replace") as f:
        print("".join(f.readlines()[-30:]))

FP16 SERVER HEALTHY: http://localhost:8000/v1/models -> 200


In [11]:
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

11581 MiB


In [12]:
from openai import OpenAI
import time

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed",
)

prompt = "Explain in simple terms what an inference server does."

start = time.time()

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200,
)

elapsed = time.time() - start
tokens = response.usage.completion_tokens

print("Completion tokens:", tokens)
print("Elapsed time:", round(elapsed, 2), "seconds")
print("Tokens/s:", round(tokens / elapsed, 2))

print("\nResponse:")
print(response.choices[0].message.content)

Completion tokens: 172
Elapsed time: 3.03 seconds
Tokens/s: 56.7

Response:
An inference server is like a smart assistant for training a model. When you train a model, you're essentially trying to teach it to understand and predict things. Once the model is trained, you want to use it to make decisions or predictions with new, unseen data.

The inference server takes the new data, processes it, and makes a prediction using the model that was trained. Just like how a teacher gives you new homework to do, the inference server gives you new predictions to make with the model it just trained.

The inference server can be programmed in different languages and perform different tasks. It can work with different kinds of data (like pictures, numbers, text) and can even be trained on its own to understand new types of data.

In summary, the inference server is like a powerful calculator that helps decision-makers make better predictions with new data.


In [13]:
SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",
    "A user asks for the weather in Riyadh and the time in Tokyo. "
    "What two tool calls would you make?",
    "Refactor this into a single sentence: The GPU was busy but not "
    "productive, because decode is memory-bound.",
    "List the steps to roll back a bad deployment, in order.",
    "Explain quantisation to a non-technical manager in three sentences.",
]

from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        messages=[{"role": "user", "content": p}],
        max_tokens=200
    )

    print("PROMPT:", p[:50], "...")
    print(r.choices[0].message.content)
    print("\n" + "="*80 + "\n")

PROMPT: Write a two-sentence summary of what an inference  ...
The inference server is responsible for executing model inference, which involves taking in data, running it through a pre-trained model, and producing relevant output.


PROMPT: A user asks for the weather in Riyadh and the time ...
To get the weather forecast for Riyadh and the current time in Tokyo, I would typically make API calls to weather and timezone services. Assuming I can make API calls, the calls I would make are:

1. A call to a weather API to get the weather forecast for Riyadh.
2. A call to a time API to get the current time in Tokyo.

The appropriate APIs for these requests would be the Weather API provider (e.g., OpenWeatherMap, AccuWeather, etc.) and the Time API provider (e.g., IANA Time Zone Database, OpenCulture/Language Code Database, etc.).


PROMPT: Refactor this into a single sentence: The GPU was  ...
The GPU was being utilized, but its productivity was low due to memory bottlenecks in the decode t

In [14]:
import os
import signal
import time

os.killpg(os.getpgid(server.pid), signal.SIGTERM)

time.sleep(3)

print("FP16 server stopped.")

FP16 server stopped.


In [15]:
import subprocess

MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

cmd = [
    "/content/venv/bin/python",
    "-m", "vllm.entrypoints.openai.api_server"
]

for k, v in SERVER_ARGS.items():
    if v is None:
        cmd.append(k)
    else:
        cmd += [k, str(v)]

print("Launching AWQ server...")
print(" ".join(cmd))

logf = open(SERVER_LOG, "wb")

server = subprocess.Popen(
    cmd,
    stdout=logf,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print(f"AWQ server started. PID: {server.pid}")

Launching AWQ server...
/content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
AWQ server started. PID: 7437


In [16]:
import time
import urllib.request
import urllib.error

url = "http://localhost:8000/v1/models"
timeout_s = 300
deadline = time.time() + timeout_s

while time.time() < deadline:
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            if r.status == 200:
                print("AWQ SERVER HEALTHY:", url, "->", r.status)
                break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass

    time.sleep(3)
else:
    print("TIMED OUT")
    print("Last 30 lines of server.log:")
    with open("/content/server.log", "r", errors="replace") as f:
        print("".join(f.readlines()[-30:]))

AWQ SERVER HEALTHY: http://localhost:8000/v1/models -> 200


In [17]:
SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",
    "A user asks for the weather in Riyadh and the time in Tokyo. "
    "What two tool calls would you make?",
    "Refactor this into a single sentence: The GPU was busy but not "
    "productive, because decode is memory-bound.",
    "List the steps to roll back a bad deployment, in order.",
    "Explain quantisation to a non-technical manager in three sentences.",
]

from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
        messages=[{"role": "user", "content": p}],
        max_tokens=200
    )

    print("PROMPT:", p[:50], "...")
    print(r.choices[0].message.content)
    print("\n" + "="*80 + "\n")

PROMPT: Write a two-sentence summary of what an inference  ...
An inference server is responsible for processing incoming requests and generating responses based on the input data, typically for tasks such as machine learning model inference, predictive analytics, and automated decision-making.


PROMPT: A user asks for the weather in Riyadh and the time ...
To fetch the weather in Riyadh and the time in Tokyo for a specific date, you would typically make two API calls. Assuming the services that provide these information are accessible through two different APIs (API One for weather and API Two for time), you would issue the following calls:

For weather in Riyadh:
```plaintext
http://api.weather.com/w/chat?apiKey=[your_api_key]&conditions=all&urls=http://weather.com/re/ (replaced "re" with Riyadh)
```

To fetch the time in Tokyo (`Tokyo`):
```plaintext
http://api.timeZone.com/timetables/v1/timeships.json?key=apikey&cityIds=891&api-language=ru (Japan time zone ID 891)
```

Replace `[y

In [18]:
# Function-calling smoke test for Lab W3D4 (quantise and lock).
# Given in full. You run it; you do not write it. Paste the whole file as one
# Colab cell (after the tool-call-enabled vLLM server is healthy), then call
# run_smoke(base_url=..., model=...).
#
# What it does: fires 3 canonical prompts, k times each for n=10 total
# attempts - 8 that want a tool call, 2 distractors that must NOT call - and
# scores each attempt's BEHAVIOUR: a valid parseable tool_calls when one is
# wanted, or a clean refusal on the distractor.
# Gate: PASS if at least 8 of 10 attempts show correct behaviour AND the
# distractor stays call-free in the majority of its attempts. A model that
# always calls a tool fails the real consumer, so restraint is scored.
#
# It talks to the OpenAI-compatible /v1 endpoint, so the same test works against
# any team's service. No secrets: the local vLLM server needs no key.

from openai import OpenAI

# Two tools the model may call. Shapes match the OpenAI tools schema.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate an arithmetic expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string",
                                   "description": "e.g. 23 * 19"},
                },
                "required": ["expression"],
            },
        },
    },
]

# The 3 canonical prompts. Each carries how many attempts (k) it gets and how many
# tool calls a correct answer makes. n = sum of k = 10.
#   two_tool:   needs BOTH tools (weather + calculator)   -> expect >= 1 call
#   single:     needs ONE tool                            -> expect >= 1 call
#   distractor: needs NO tool, must NOT call one          -> expect 0 calls
CANONICAL = [
    {
        "id": "two_tool",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Riyadh, and what is 23 multiplied "
                  "by 19? Use your tools.",
    },
    {
        "id": "single",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Tokyo right now? Use your tools.",
    },
    {
        "id": "distractor",
        "k": 2,
        "wants_call": False,
        "prompt": "In one sentence, explain what a tool call is. Do not call "
                  "any tool; just answer.",
    },
]


def _tool_calls_of(message) -> list:
    """Return the parsed tool_calls list on a response message, or []."""
    tc = getattr(message, "tool_calls", None)
    return list(tc) if tc else []


def _valid_call(call) -> bool:
    """A tool call is valid if it names a known function and its arguments
    parse as JSON with the required field present."""
    import json
    try:
        fn = call.function.name
        if fn not in ("get_weather", "calculate"):
            return False
        args = json.loads(call.function.arguments or "{}")
    except (AttributeError, ValueError):
        return False
    if fn == "get_weather":
        return isinstance(args.get("city"), str) and bool(args["city"])
    if fn == "calculate":
        return isinstance(args.get("expression"), str) and bool(args["expression"])
    return False


def run_smoke(base_url: str, model: str, temperature: float = 0.0) -> dict:
    """Run the smoke test. Returns a result dict with counts and the pass gate."""
    client = OpenAI(base_url=base_url, api_key="not-needed")

    total_attempts = 0
    valid_call_attempts = 0          # attempts that returned >=1 valid tool call
    distractor_attempts = 0
    distractor_call_free = 0         # distractor attempts that made NO tool call
    per_prompt = {}

    for spec in CANONICAL:
        pid, k, wants = spec["id"], spec["k"], spec["wants_call"]
        got_valid = 0
        got_call_free = 0
        for _ in range(k):
            total_attempts += 1
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": spec["prompt"]}],
                tools=TOOLS,
                tool_choice="auto",
                temperature=temperature,
                max_tokens=256,
            )
            msg = resp.choices[0].message
            calls = _tool_calls_of(msg)
            any_valid = any(_valid_call(c) for c in calls)

            if wants:
                # a "wants a call" prompt counts toward the 8/10 gate when it
                # returns at least one valid tool call
                if any_valid:
                    valid_call_attempts += 1
                    got_valid += 1
            else:
                # the distractor counts toward the 8/10 gate when it correctly
                # makes NO tool call, and separately toward distractor compliance
                distractor_attempts += 1
                if not calls:
                    valid_call_attempts += 1
                    distractor_call_free += 1
                    got_call_free += 1

        per_prompt[pid] = {"k": k, "wants_call": wants,
                           "valid": got_valid, "call_free": got_call_free}

    # gate: >=8/10 correct behaviours AND distractor call-free in the majority
    distractor_majority = (distractor_call_free * 2 > distractor_attempts) \
        if distractor_attempts else True
    passed = (valid_call_attempts >= 8) and distractor_majority

    return {
        "model": model,
        "total_attempts": total_attempts,       # 10
        "score": valid_call_attempts,           # correct behaviours, out of 10
        "distractor_attempts": distractor_attempts,
        "distractor_call_free": distractor_call_free,
        "distractor_majority_clean": distractor_majority,
        "per_prompt": per_prompt,
        "passed": passed,
    }


In [19]:
result = run_smoke(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct-AWQ"
)

print(result)

{'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ', 'total_attempts': 10, 'score': 10, 'distractor_attempts': 2, 'distractor_call_free': 2, 'distractor_majority_clean': True, 'per_prompt': {'two_tool': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'single': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'distractor': {'k': 2, 'wants_call': False, 'valid': 0, 'call_free': 2}}, 'passed': True}


In [20]:
import json

with open("smoke_result.json", "w") as f:
    json.dump(result, f, indent=2)

print("smoke_result.json saved")

smoke_result.json saved


In [22]:
content = """# Model lock (team record)

## The locked model

- Model id: Qwen/Qwen2.5-1.5B-Instruct-AWQ
- Quantisation: awq
- Why this one: Passed the function-calling smoke test with 10/10 while providing strong VRAM headroom.

## The launch flags

--model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096
--gpu-memory-utilization 0.85
--quantization awq --enable-auto-tool-choice --tool-call-parser hermes

- Tool-call parser: hermes

## The smoke score

- Score (valid behaviours out of 10): 10
- Distractor stayed call-free in the majority: yes
- Passed the gate (>= 8/10 and distractor majority clean): yes
- Measured against: AWQ — 10/10

## Quality spot check note

The AWQ build generally held up on the five-prompt side-by-side check, with good performance on summarisation, refactoring, and rollback prompts. It showed some degradation in instruction following and explanation quality on the tool-call and quantisation prompts compared with FP16, but it passed the official function-calling smoke test with 10/10.
"""

with open("model-lock.md", "w") as f:
    f.write(content)

print("model-lock.md saved")

model-lock.md saved


In [23]:
# Green-check verifier for Lab W3D4 (quantise and lock).
# Paste this as the last cell of your day-4 notebook and run it. It reads
# smoke_result.json (written from the smoke test) and model-lock.md, and checks
# that the smoke score meets the gate and that the lock file is fully filled in.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os, re


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) smoke result
    if not os.path.exists("smoke_result.json"):
        fail("smoke_result.json not found; write it in Cell 5")
    try:
        with open("smoke_result.json") as fh:
            result = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"smoke_result.json is not valid JSON: {exc}")

    for key in ("score", "total_attempts", "distractor_majority_clean", "passed"):
        if key not in result:
            fail(f"smoke_result.json missing key: {key}")

    score = result["score"]
    total = result["total_attempts"]
    if not isinstance(score, int) or not isinstance(total, int):
        fail("score and total_attempts must be integers")
    if total != 10:
        fail(f"total_attempts is {total}, the smoke test defines n=10")
    if score < 8:
        fail(f"smoke score {score}/10 is below the 8/10 gate")
    if not result["distractor_majority_clean"]:
        fail("distractor did not stay call-free in the majority; a model that "
             "always calls a tool fails the real consumer")
    if not result["passed"]:
        fail("smoke test reports passed=false")

    # 2) model-lock.md fully filled in
    if not os.path.exists("model-lock.md"):
        fail("model-lock.md not found")
    with open("model-lock.md") as fh:
        lock = fh.read()
    remaining = re.findall(r"FILL:", lock)
    if remaining:
        fail(f"model-lock.md has {len(remaining)} unfilled FILL: placeholders")
    # require a concrete model id line
    if not re.search(r"Model id:\s*\S+", lock):
        fail("model-lock.md has no concrete Model id")

    print(f"smoke score: {score}/{total}, distractor clean: "
          f"{result['distractor_majority_clean']}")
    print("model-lock.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


smoke score: 10/10, distractor clean: True
model-lock.md: all fields filled
GREEN CHECK: PASS
